In [48]:
import json
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "amici2014lack")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Amici_2014_ProcRSocB_LAKS_exp1.csv")
complete_path_2 = os.path.join(original_data_pathway, "Amici_2014_ProcRSocB_LAKS_exp2.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [49]:
import pandas as pd
import numpy as np
df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)
df1['experiment_name']="platforms_task"

df1 = df1.rename(columns={"Subject (in A/B)": "participant", 
    "Partner (in C)": "participant_2", 
    "Choice made: Prosocial1_Selfish0": "choice_prosocial1_selfish0",
    "Prosocial choice_1, Selfish choice_0": "choice_prosocial1_selfish0"})
df1['role']="subject_in_a/b"
df1['role_2']="partner_in_c"
# df1.columns


In [50]:

df2=df2.rename(columns={"Subject": "participant",
    "Prosocial choice_1, Selfish choice_0":"choice_prosocial1_selfish0",
    "Partner":"participant_2"})
df2['experiment_name']="tokens_task"
df2['role']="subject"
df2['role_2']="partner"
# df2.columns

In [51]:


comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
# df2.columns

In [52]:
df2.dropna(subset=['participant'], inplace=True)
df2['Date']= pd.to_datetime(df2['Date'],format='%d.%m.%Y')
df2['year']= df2['Date'].dt.year
df2['month']= df2['Date'].dt.month
df2['day']= df2['Date'].dt.day

df2['year']= df2['year'].astype(int)
df2['month']= df2['month'].astype(int)
df2['day']= df2['day'].astype(int)




In [53]:
df2=df2.applymap(lambda s: s.lower() if type(s) == str else s) 
df2['participant'] = df2['participant'].str.rstrip()
df2['participant_2'] = df2['participant_2'].str.rstrip()

for x,y in zip(df_name['wrong'],df_name['right']):
    df2['participant'].replace(x, y, inplace=True)
    df2['participant_2'].replace(x, y, inplace=True)


In [54]:

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
df2= df2.merge(ape_dob,left_on='participant', right_on='name', how='left')


comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
df2= df2.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
df2['name'].unique()

array(['frodo', 'lome', 'tai', 'fraukje', 'fifi', 'gertrudia', 'jahaga',
       'yasa', 'gemena', 'lexi', 'fimi', 'luiza', 'viringika', 'kibara',
       'abeeku', 'bimbo', 'dokana', 'padana', 'pini', 'kuno', 'jasongo',
       'kumili'], dtype=object)

In [55]:
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    df2[x] = df2['year'].astype(str) + '-' + df2['month'].astype(str) + '-' + df2['day'].astype(str)
    df2[x] = pd.to_datetime(df2[x])
    df2[y] = pd.to_datetime(df2[y])
    df2[k] = (df2[x] - df2[y]).dt.days//365

In [56]:
df2['age_in_years_2'] = df2['age_in_years_2'].astype(str)
for index, row in df2.iterrows():
    if row['Condition'] == 'token preference':
        df2.at[index,'participant_2'] = ''
        df2.at[index,'role_2'] = ''
        df2.at[index,'age_in_years_2'] = ''

In [57]:
data_frames=[df1, df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"species": "species_original"})
    x['study_id']="amici2014lack"
    x['participant'] = x['participant'].str.rstrip()
    x['participant_2'] = x['participant_2'].str.rstrip()
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [58]:

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)
    fulldf['participant_2'].replace(x, y, inplace=True)


In [59]:
fulldf['participant'].replace('', np.nan, inplace=True)
fulldf.dropna(subset=['participant'], inplace=True)


comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)  
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')


comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
fulldf= fulldf.merge(apedf_2,left_on='participant_2', right_on='name_2', how='left')

In [60]:
fulldf['participant_2'].replace('', np.nan, inplace=True)
dyad=[]
for index, row in fulldf.iterrows():
    if not pd.isna(row['participant_2']):
        dyad.append(row['participant'] + '_' + row['participant_2'])
    else:
        dyad.append("")
fulldf = fulldf.assign(dyad=dyad)

In [61]:
fulldf['condition'].replace(' ', '_', inplace=True, regex=True)
fulldf['tokens given'].replace(' ', '_', inplace=True, regex=True)

fulldf=fulldf.rename(columns={"token left": "token_left", 
    "token right": "token_right",
    "tokens given": "tokens_given",
    "token chosen in training and token preference": "token_chosen_in_training_and_token_preference"})


In [62]:
fulldf['role'].replace('subject', 'focal_participant', inplace=True, regex=True)
# fulldf['role'].unique()

In [63]:
amici2014lack_standardized=fulldf[['study_id', 'experiment_name', 'year', 'month', 'day', 
        'participant', 'age_in_years','sex', 'role', 'participant_2','age_in_years_2', 'sex_2', 'role_2', 'species','dyad', 'session', 'trial', 'condition',
       'choice_prosocial1_selfish0', 
        'token_left', 'token_right', 'tokens_given',
       'token_chosen_in_training_and_token_preference']]

In [64]:
fulldf['experiment_name'].unique()

array(['platforms_task', 'tokens_task'], dtype=object)

In [65]:
exp1 = amici2014lack_standardized[amici2014lack_standardized['experiment_name'] == 'platforms_task']
exp2 = amici2014lack_standardized[amici2014lack_standardized['experiment_name'] == 'tokens_task']

experiments = [[exp1, 'amici2014lack_exp1'], 
                [ exp2, 'amici2014lack_exp2']]

for x,y in experiments:
    x = x.dropna(axis=1, how='all')
    comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
    x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
    ##glossaries
    names = x.columns.tolist()
    df = pd.DataFrame(names)
    df = df.rename(columns={0: "column_name"})
    df["description"] = ""
    studyID_glossary=df[["column_name", "description"]]

    comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
    studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)